# Predicting Credit Card Fraud — Playground Series S6E11 (v2)

**v1 result:** leaderboard ROC AUC 0.94650, OOF blend 0.94624.
**v2 changes vs v1:**
1. New features: cyclical hour / day-of-week encoding, frequency encoding
   of the 5 categoricals, and extra interactions (amount x risk, amount x
   velocity, balance per card-age month, foreign x VPN).
2. Seed averaging: LightGBM and XGBoost trained with 2 seeds each
   (CatBoost keeps 1 seed — it is the slowest model), which reduces the
   variance of the blend.
3. Same 5-fold stratified CV and the same LightGBM + XGBoost + CatBoost
   rank-averaged blend, so v2 stays directly comparable with v1.


In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

DATA = '/kaggle/input/playground-series-s6e11'
if not os.path.exists(DATA):
    DATA = '/kaggle/input/competitions/playground-series-s6e11'
print('data dir:', DATA)
train = pd.read_csv(f'{DATA}/train.csv')
test = pd.read_csv(f'{DATA}/test.csv')
print('train:', train.shape, '| test:', test.shape)
print('fraud rate in train: %.4f' % train['is_fraud'].mean())

TARGET = 'is_fraud'
CATS = ['merchant_category', 'card_type', 'auth_method', 'channel', 'device_type']
BIN = ['is_foreign_transaction', 'is_new_merchant', 'used_vpn',
       'ip_country_mismatch', 'billing_shipping_mismatch', 'is_ai_generated_scam_attempt']


In [ ]:
def add_features(train, test):
    tr, te = train.copy(), test.copy()
    # Frequency encoding (no target involved, so fitting on train+test is safe)
    combined = pd.concat([tr[CATS], te[CATS]], axis=0)
    for c in CATS:
        freq = combined[c].value_counts(normalize=True)
        tr[c + '_freq'] = tr[c].map(freq)
        te[c + '_freq'] = te[c].map(freq)
    for df in (tr, te):
        df['log_amount'] = np.log1p(df['amount_usd'])
        df['log_balance'] = np.log1p(df['account_balance_usd'])
        df['log_distance'] = np.log1p(df['distance_from_home_km'])
        df['amount_per_balance'] = df['amount_usd'] / (df['account_balance_usd'] + 1)
        df['amount_vs_age'] = df['amount_usd'] / (df['customer_age'] + 1)
        df['balance_per_month'] = df['account_balance_usd'] / (df['card_age_months'] + 1)
        df['txn_velocity'] = df['txn_count_last_24h'] * df['velocity_score']
        df['risk_combo'] = df['merchant_risk_score'] * (df['cvv_retry_count'] + 1)
        df['suspicious_flags'] = df[BIN].sum(axis=1)
        df['night_txn'] = ((df['time_of_day_hour'] < 6) | (df['time_of_day_hour'] > 22)).astype(int)
        df['amount_x_foreign'] = df['amount_usd'] * df['is_foreign_transaction']
        df['amount_x_new_merchant'] = df['amount_usd'] * df['is_new_merchant']
        df['amount_x_risk'] = df['amount_usd'] * df['merchant_risk_score']
        df['amount_x_velocity'] = df['amount_usd'] * df['velocity_score']
        df['foreign_x_vpn'] = df['is_foreign_transaction'] * df['used_vpn']
        df['disputes_x_risk'] = df['prior_disputes'] * df['merchant_risk_score']
        df['hour_sin'] = np.sin(2 * np.pi * df['time_of_day_hour'] / 24)
        df['hour_cos'] = np.cos(2 * np.pi * df['time_of_day_hour'] / 24)
        df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
        df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
    return tr, te

train, test = add_features(train, test)
FEATURES = [c for c in train.columns if c not in ('id', TARGET)]
print('n features:', len(FEATURES))


In [ ]:
# ---- 5-fold stratified CV, seed-averaged ----
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
X, y = train[FEATURES], train[TARGET]
X_test = test[FEATURES]
X_oh = pd.get_dummies(X, columns=CATS)
Xt_oh = pd.get_dummies(X_test, columns=CATS).reindex(columns=X_oh.columns, fill_value=0)

oof_lgb = np.zeros(len(X)); oof_xgb = np.zeros(len(X)); oof_cat = np.zeros(len(X))
pred_lgb = np.zeros(len(X_test)); pred_xgb = np.zeros(len(X_test)); pred_cat = np.zeros(len(X_test))

def run_lgb(seed):
    global oof_lgb, pred_lgb
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        Xl_tr, Xl_va = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy()
        for c in CATS:
            Xl_tr[c] = Xl_tr[c].astype('category')
            Xl_va[c] = Xl_va[c].astype('category')
        Xl_te = X_test.copy()
        for c in CATS:
            Xl_te[c] = Xl_te[c].astype('category')
        m = lgb.LGBMClassifier(n_estimators=1500, learning_rate=0.05, num_leaves=63,
                               feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
                               min_child_samples=100, random_state=seed, n_jobs=-1, verbose=-1)
        m.fit(Xl_tr, y.iloc[tr_idx], eval_set=[(Xl_va, y.iloc[va_idx])],
              callbacks=[lgb.early_stopping(100, verbose=False)])
        oof_lgb[va_idx] += m.predict_proba(Xl_va)[:, 1] / 2
        pred_lgb += m.predict_proba(Xl_te)[:, 1] / (5 * 2)

def run_xgb(seed):
    global oof_xgb, pred_xgb
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        mx = xgb.XGBClassifier(n_estimators=1500, learning_rate=0.05, max_depth=7,
                               subsample=0.8, colsample_bytree=0.8, eval_metric='auc',
                               early_stopping_rounds=100, random_state=seed, n_jobs=-1)
        mx.fit(X_oh.iloc[tr_idx], y.iloc[tr_idx],
               eval_set=[(X_oh.iloc[va_idx], y.iloc[va_idx])], verbose=False)
        oof_xgb[va_idx] += mx.predict_proba(X_oh.iloc[va_idx])[:, 1] / 2
        pred_xgb += mx.predict_proba(Xt_oh)[:, 1] / (5 * 2)

def run_cat(seed):
    global oof_cat, pred_cat
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        mc = CatBoostClassifier(iterations=1500, learning_rate=0.05, depth=7,
                                loss_function='Logloss', eval_metric='AUC',
                                od_type='Iter', od_wait=100, random_seed=seed, verbose=0)
        mc.fit(X.iloc[tr_idx], y.iloc[tr_idx], cat_features=CATS,
               eval_set=(X.iloc[va_idx], y.iloc[va_idx]), use_best_model=True)
        oof_cat[va_idx] += mc.predict_proba(X.iloc[va_idx])[:, 1]
        pred_cat += mc.predict_proba(X_test)[:, 1] / 5

for s in (42, 7):
    print(f'== LightGBM seed {s} ==')
    run_lgb(s)
for s in (42, 7):
    print(f'== XGBoost seed {s} ==')
    run_xgb(s)
print('== CatBoost seed 42 ==')
run_cat(42)

def rank_avg(*preds):
    return np.mean([rankdata(p) / len(p) for p in preds], axis=0)

oof_blend = rank_avg(oof_lgb, oof_xgb, oof_cat)
print('OOF AUC | LGBM %.5f | XGB %.5f | CAT %.5f | BLEND %.5f' % (
    roc_auc_score(y, oof_lgb), roc_auc_score(y, oof_xgb),
    roc_auc_score(y, oof_cat), roc_auc_score(y, oof_blend)))
print('v1 OOF blend was 0.94624 — compare on the leaderboard, not on OOF alone')


In [ ]:
# ---- Final submission: plain average of the three models' test probabilities ----
final_pred = np.clip((pred_lgb + pred_xgb + pred_cat) / 3, 0, 1)
submission = pd.DataFrame({'id': test['id'], 'is_fraud': final_pred})
submission.to_csv('submission.csv', index=False)
print(submission.head())
print('submission saved:', submission.shape)
print('pred stats:', submission['is_fraud'].describe())
